In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/anrf-aise-hack-pha

In [2]:
# =============================================================================
# CELL 1 - Imports, config, device, split helpers
# =============================================================================

import gc
import json
import math
import os
import shutil
import time
import warnings
from contextlib import nullcontext
from copy import deepcopy
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
torch.set_num_threads(1)
torch.backends.cudnn.benchmark = True
if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
    torch.backends.cuda.matmul.allow_tf32 = True
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("medium")

RUN_TUNE = True
RUN_FINAL_TRAIN = True
RUN_INFER = True
DEBUG = False
FORCE_SINGLE_GPU = True
VAL_WINDOWS_PER_MONTH = 64
TOPK_CKPTS = 4


class CFG:
    ROOT = (
        "/kaggle/input/competitions/"
        "anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/"
        "aisehack-theme-2"
    )
    RAW_ROOT = f"{ROOT}/raw"
    TEST_ROOT = f"{ROOT}/test_in"
    LAT_LON_PATH = f"{RAW_ROOT}/lat_long.npy"

    OUT_DIR = "/kaggle/working/aisehack_finetuned_latest"
    TUNE_CKPT_DIR = f"{OUT_DIR}/tune_checkpoints"
    FINAL_CKPT_DIR = f"{OUT_DIR}/final_checkpoints"
    LOG_PATH = f"{OUT_DIR}/log.json"
    STATS_TUNE_PATH = f"{OUT_DIR}/stats_tune.json"
    STATS_FULL_PATH = f"{OUT_DIR}/stats_full.json"
    HOTSPOT_TUNE_PATH = f"{OUT_DIR}/hotspot_prior_tune.npy"
    HOTSPOT_FULL_PATH = f"{OUT_DIR}/hotspot_prior_full.npy"
    PREDS_PATH = "/kaggle/working/preds.npy"

    MONTHS = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]

    H = 140
    W = 124
    PAD_H = 144
    PAD_W = 128
    TIME_IN = 10
    TIME_OUT = 16
    HORIZON = TIME_IN + TIME_OUT
    STRIDE = 1

    BASE_FEATURES = [
        "cpm25",
        "q2",
        "t2",
        "u10",
        "v10",
        "swdown",
        "pblh",
        "psfc",
        "rain",
        "PM25",
        "NH3",
        "SO2",
        "NOx",
        "NMVOC_e",
        "NMVOC_finn",
        "bio",
    ]
    DERIVED_FEATURES = [
        "wind_speed",
        "wind_divergence",
        "ventilation_index",
        "pblh_inverse",
    ]
    DYNAMIC_FEATURES = BASE_FEATURES + DERIVED_FEATURES
    STATIC_FEATURES = [
        "lat",
        "lon",
        "cpm25_mean_10",
        "cpm25_trend_3",
        "cpm25_std_10",
        "hotspot_prior",
    ]

    TARGET_IDX = 0
    N_DYNAMIC = len(DYNAMIC_FEATURES)
    N_STATIC = len(STATIC_FEATURES)

    EMISSION_FEATURES = {
        "PM25",
        "NH3",
        "SO2",
        "NOx",
        "NMVOC_e",
        "NMVOC_finn",
        "bio",
    }
    WIND_TANH_FEATURES = {"u10", "v10"}

    EPS = 1e-6
    EPISODE_THRESHOLD_STD = 1.5
    HOTSPOT_KERNEL = 5
    HOTSPOT_WEIGHT_SCALE = 1.5

    MODEL_BASE = 48
    MODEL_HIDDEN = 160
    DROPOUT = 0.10

    TRAIN_BATCH = 4
    GRAD_ACCUM = 2
    INFER_BATCH = 4
    NUM_WORKERS = 0
    EMA_DECAY = 0.995

    SEED = 42
    FINAL_SEEDS = [42, 3407]
    TUNE_EPOCHS = 10
    FINAL_EPOCHS = None
    PATIENCE = 3
    WARMUP_EPOCHS = 1
    FINAL_EXTRA_EPOCHS = 1
    FINAL_SNAPSHOT_OFFSETS = (-1, 0, 1)

    LR = 8e-4
    FINAL_LR = 6e-4
    WEIGHT_DECAY = 1e-4

    LOSS_WEIGHTS = {
        "delta": 0.30,
        "global_smape": 0.25,
        "episode_smape": 0.18,
        "episode_under": 0.12,
        "hotspot": 0.10,
        "corr": 0.05,
    }


os.makedirs(CFG.OUT_DIR, exist_ok=True)
os.makedirs(CFG.TUNE_CKPT_DIR, exist_ok=True)
os.makedirs(CFG.FINAL_CKPT_DIR, exist_ok=True)

if torch.cuda.is_available():
    if FORCE_SINGLE_GPU:
        DEVICE = torch.device("cuda:0")
        torch.cuda.set_device(0)
    else:
        DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
AMP_ENABLED = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CFG.SEED)

print("=" * 80)
print("AISEHack Fine-Tuned Latest Notebook")
print(f"Device            : {DEVICE}")
print(f"GPUs detected     : {N_GPUS}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU         : {props.name} ({props.total_memory / 1e9:.1f} GB)")
print("DataParallel      : OFF")
print(f"AMP               : {'ON' if AMP_ENABLED else 'OFF'}")
print(f"DEBUG             : {DEBUG}")
print(f"VAL windows/month : {VAL_WINDOWS_PER_MONTH}")
print(f"Top-K tune ckpts  : {TOPK_CKPTS}")
print(f"Final seeds       : {CFG.FINAL_SEEDS}")
print("=" * 80)


def month_length(month: str) -> int:
    path = os.path.join(CFG.RAW_ROOT, month, "cpm25.npy")
    return int(np.load(path, mmap_mode="r").shape[0])


def build_split_items():
    train_items = []
    val_items = []
    all_items = []
    train_cutoffs = {}
    window_counts = {}

    for month in CFG.MONTHS:
        total_steps = month_length(month)
        starts = list(range(0, total_steps - CFG.HORIZON + 1, CFG.STRIDE))
        if len(starts) <= VAL_WINDOWS_PER_MONTH:
            raise ValueError(f"{month} has only {len(starts)} windows; cannot hold out {VAL_WINDOWS_PER_MONTH}")

        train_starts = starts[:-VAL_WINDOWS_PER_MONTH]
        val_starts = starts[-VAL_WINDOWS_PER_MONTH:]
        train_cutoffs[month] = total_steps - VAL_WINDOWS_PER_MONTH
        window_counts[month] = len(starts)

        train_items.extend({"month": month, "start": start} for start in train_starts)
        val_items.extend({"month": month, "start": start} for start in val_starts)
        all_items.extend({"month": month, "start": start} for start in starts)

        print(
            f"{month}: total_windows={len(starts)} | "
            f"train_windows={len(train_starts)} | val_windows={len(val_starts)}"
        )

    return train_items, val_items, all_items, train_cutoffs, window_counts


TRAIN_ITEMS, VAL_ITEMS, ALL_ITEMS, TRAIN_CUTOFFS, WINDOW_COUNTS = build_split_items()

if DEBUG:
    TRAIN_ITEMS = TRAIN_ITEMS[: max(CFG.TRAIN_BATCH * CFG.GRAD_ACCUM * 4, 16)]
    VAL_ITEMS = VAL_ITEMS[: max(CFG.TRAIN_BATCH * 2, 8)]
    ALL_ITEMS = ALL_ITEMS[: max(CFG.TRAIN_BATCH * CFG.GRAD_ACCUM * 5, 32)]


def load_lat_lon() -> tuple[np.ndarray, np.ndarray]:
    arr = np.load(CFG.LAT_LON_PATH).astype(np.float32)
    if arr.shape == (2, CFG.H, CFG.W):
        lat, lon = arr[0], arr[1]
    elif arr.shape == (CFG.H, CFG.W, 2):
        lat, lon = arr[..., 0], arr[..., 1]
    else:
        raise ValueError(f"Unexpected lat_long.npy shape: {arr.shape}")

    lat = (lat - lat.min()) / (lat.max() - lat.min() + CFG.EPS)
    lon = (lon - lon.min()) / (lon.max() - lon.min() + CFG.EPS)
    return lat.astype(np.float32), lon.astype(np.float32)


LAT_GRID, LON_GRID = load_lat_lon()

AISEHack Fine-Tuned Latest Notebook
Device            : cuda:0
GPUs detected     : 2
Using GPU         : Tesla T4 (15.6 GB)
DataParallel      : OFF
AMP               : ON
DEBUG             : False
VAL windows/month : 64
Top-K tune ckpts  : 4
Final seeds       : [42, 3407]
APRIL_16: total_windows=690 | train_windows=626 | val_windows=64
JULY_16: total_windows=714 | train_windows=650 | val_windows=64
OCT_16: total_windows=714 | train_windows=650 | val_windows=64
DEC_16: total_windows=714 | train_windows=650 | val_windows=64


In [3]:
# =============================================================================
# CELL 2 - Data processing and artifact builders
# =============================================================================


def load_month_base_arrays(month: str) -> Dict[str, np.ndarray]:
    arrays = {}
    for feature in CFG.BASE_FEATURES:
        arrays[feature] = np.load(os.path.join(CFG.RAW_ROOT, month, f"{feature}.npy")).astype(np.float32)
    return arrays


def load_test_base_arrays_memmap() -> Dict[str, np.memmap]:
    arrays = {}
    for feature in CFG.BASE_FEATURES:
        arrays[feature] = np.load(os.path.join(CFG.TEST_ROOT, f"{feature}.npy"), mmap_mode="r")
    return arrays


def compute_derived_features(base_arrays: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    u10 = base_arrays["u10"]
    v10 = base_arrays["v10"]
    pblh = base_arrays["pblh"]

    wind_speed = np.sqrt(np.maximum(u10 * u10 + v10 * v10, 0.0)).astype(np.float32)
    dudx = np.gradient(u10, axis=-1).astype(np.float32)
    dvdy = np.gradient(v10, axis=-2).astype(np.float32)
    wind_divergence = (dudx + dvdy).astype(np.float32)
    ventilation_index = (wind_speed * np.maximum(pblh, 0.0)).astype(np.float32)
    pblh_inverse = (1.0 / np.maximum(pblh, 1.0)).astype(np.float32)

    return {
        "wind_speed": wind_speed,
        "wind_divergence": wind_divergence,
        "ventilation_index": ventilation_index,
        "pblh_inverse": pblh_inverse,
    }


def compute_stats(train_cutoffs: Dict[str, int] | None = None) -> Dict[str, Dict[str, float]]:
    accum = {
        feature: {"sum": 0.0, "sum_sq": 0.0, "count": 0}
        for feature in CFG.DYNAMIC_FEATURES
    }

    desc = "Computing full-data stats" if train_cutoffs is None else "Computing tune stats"
    for month in tqdm(CFG.MONTHS, desc=desc):
        cutoff = train_cutoffs[month] if train_cutoffs is not None else None
        base_arrays = load_month_base_arrays(month)
        all_arrays = {**base_arrays, **compute_derived_features(base_arrays)}
        for feature in CFG.DYNAMIC_FEATURES:
            arr = all_arrays[feature] if cutoff is None else all_arrays[feature][:cutoff]
            accum[feature]["sum"] += float(arr.sum(dtype=np.float64))
            accum[feature]["sum_sq"] += float(np.square(arr, dtype=np.float64).sum(dtype=np.float64))
            accum[feature]["count"] += int(arr.size)
        del base_arrays, all_arrays
        gc.collect()

    stats = {}
    for feature, values in accum.items():
        mean = values["sum"] / values["count"]
        var = max(values["sum_sq"] / values["count"] - mean * mean, 1e-8)
        stats[feature] = {
            "mean": float(mean),
            "std": float(math.sqrt(var) + CFG.EPS),
        }
    return stats


def normalize_feature(arr: np.ndarray, feature: str, stats: Dict[str, Dict[str, float]]) -> np.ndarray:
    out = (arr.astype(np.float32) - stats[feature]["mean"]) / stats[feature]["std"]
    if feature in CFG.WIND_TANH_FEATURES:
        out = np.tanh(out)
    if feature in CFG.EMISSION_FEATURES:
        out = np.clip(out, -5.0, 5.0)
    return out.astype(np.float32, copy=False)


def build_month_cache(stats: Dict[str, Dict[str, float]]) -> Dict[str, np.ndarray]:
    cache = {}
    total_gb = 0.0
    for month in CFG.MONTHS:
        base_arrays = load_month_base_arrays(month)
        all_arrays = {**base_arrays, **compute_derived_features(base_arrays)}
        stacked = [normalize_feature(all_arrays[feature], feature, stats) for feature in CFG.DYNAMIC_FEATURES]
        tensor = np.stack(stacked, axis=-1).astype(np.float32, copy=False)
        cache[month] = np.ascontiguousarray(tensor)
        total_gb += cache[month].nbytes / 1e9
        print(f"Cached {month}: {cache[month].shape} | {cache[month].nbytes / 1e9:.2f} GB")
        del base_arrays, all_arrays, stacked, tensor
        gc.collect()
    print(f"Cache RAM footprint: {total_gb:.2f} GB")
    return cache


def denorm_cpm25_np(x: np.ndarray, stats: Dict[str, Dict[str, float]]) -> np.ndarray:
    return x * stats["cpm25"]["std"] + stats["cpm25"]["mean"]


def denorm_cpm25_torch(x: torch.Tensor, stats: Dict[str, Dict[str, float]]) -> torch.Tensor:
    return x * stats["cpm25"]["std"] + stats["cpm25"]["mean"]


def build_hotspot_prior(
    month_cache: Dict[str, np.ndarray],
    items: List[Dict],
    stats: Dict[str, Dict[str, float]],
) -> np.ndarray:
    accum = np.zeros((CFG.H, CFG.W), dtype=np.float64)
    total_frames = 0
    subset = items if not DEBUG else items[: min(len(items), 128)]

    for item in tqdm(subset, desc="Building hotspot prior"):
        tensor = month_cache[item["month"]][:, :, :, CFG.TARGET_IDX]
        start = item["start"]
        window = tensor[start : start + CFG.HORIZON]
        input_seq = denorm_cpm25_np(window[: CFG.TIME_IN], stats)
        target_seq = denorm_cpm25_np(window[CFG.TIME_IN :], stats)

        baseline = input_seq.mean(axis=0)
        spread = input_seq.std(axis=0)
        threshold = baseline + CFG.EPISODE_THRESHOLD_STD * spread
        mask = target_seq > threshold[None, :, :]
        accum += mask.sum(axis=0, dtype=np.int64)
        total_frames += mask.shape[0]

    prior = accum / max(total_frames, 1)
    prior_t = torch.from_numpy(prior.astype(np.float32))[None, None]
    prior_t = F.avg_pool2d(
        prior_t,
        kernel_size=CFG.HOTSPOT_KERNEL,
        stride=1,
        padding=CFG.HOTSPOT_KERNEL // 2,
    )
    prior = prior_t.squeeze(0).squeeze(0).numpy()
    prior = (prior - prior.min()) / (prior.max() - prior.min() + CFG.EPS)
    return prior.astype(np.float32)


def save_json(path: str, obj: dict) -> None:
    with open(path, "w") as handle:
        json.dump(obj, handle, indent=2)

In [4]:
# =============================================================================
# CELL 3 - Datasets, losses, metrics
# =============================================================================


class WindowDataset(Dataset):
    def __init__(self, month_cache: Dict[str, np.ndarray], items: List[Dict], hotspot_prior: np.ndarray):
        self.month_cache = month_cache
        self.items = items
        self.hotspot_prior = hotspot_prior.astype(np.float32)

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int):
        item = self.items[idx]
        tensor = self.month_cache[item["month"]]
        start = item["start"]
        window = tensor[start : start + CFG.HORIZON]

        x_dyn = np.ascontiguousarray(window[: CFG.TIME_IN], dtype=np.float32)
        y = np.ascontiguousarray(window[CFG.TIME_IN :, :, :, CFG.TARGET_IDX].transpose(1, 2, 0))
        input_cpm25_seq = np.ascontiguousarray(x_dyn[:, :, :, CFG.TARGET_IDX], dtype=np.float32)
        last_cpm25 = np.ascontiguousarray(input_cpm25_seq[-1:, :, :].transpose(1, 2, 0))

        cpm25_mean_10 = input_cpm25_seq.mean(axis=0).astype(np.float32)
        cpm25_trend_3 = ((input_cpm25_seq[-1] - input_cpm25_seq[-4]) / 3.0).astype(np.float32)
        cpm25_std_10 = input_cpm25_seq.std(axis=0).astype(np.float32)
        x_static = np.stack(
            [
                LAT_GRID,
                LON_GRID,
                cpm25_mean_10,
                cpm25_trend_3,
                cpm25_std_10,
                self.hotspot_prior,
            ],
            axis=-1,
        ).astype(np.float32)

        return (
            torch.from_numpy(x_dyn),
            torch.from_numpy(x_static),
            torch.from_numpy(y),
            torch.from_numpy(last_cpm25),
            torch.from_numpy(input_cpm25_seq),
        )


class TestStreamingDataset(Dataset):
    def __init__(self, stats: Dict[str, Dict[str, float]], hotspot_prior: np.ndarray):
        self.stats = stats
        self.hotspot_prior = hotspot_prior.astype(np.float32)
        self.arrs = load_test_base_arrays_memmap()
        self.length = int(next(iter(self.arrs.values())).shape[0])

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx: int):
        base = {feature: self.arrs[feature][idx].astype(np.float32) for feature in CFG.BASE_FEATURES}
        derived = compute_derived_features(base)
        all_features = {**base, **derived}

        x_dyn = np.stack(
            [normalize_feature(all_features[feature], feature, self.stats) for feature in CFG.DYNAMIC_FEATURES],
            axis=-1,
        ).astype(np.float32)

        input_cpm25_seq = x_dyn[:, :, :, CFG.TARGET_IDX]
        last_cpm25 = input_cpm25_seq[-1:, :, :].transpose(1, 2, 0).astype(np.float32)
        cpm25_mean_10 = input_cpm25_seq.mean(axis=0).astype(np.float32)
        cpm25_trend_3 = ((input_cpm25_seq[-1] - input_cpm25_seq[-4]) / 3.0).astype(np.float32)
        cpm25_std_10 = input_cpm25_seq.std(axis=0).astype(np.float32)
        x_static = np.stack(
            [
                LAT_GRID,
                LON_GRID,
                cpm25_mean_10,
                cpm25_trend_3,
                cpm25_std_10,
                self.hotspot_prior,
            ],
            axis=-1,
        ).astype(np.float32)

        return (
            torch.from_numpy(np.ascontiguousarray(x_dyn)),
            torch.from_numpy(np.ascontiguousarray(x_static)),
            torch.from_numpy(np.ascontiguousarray(last_cpm25)),
            torch.tensor(idx, dtype=torch.long),
        )


def pad_tensor_2d(x: torch.Tensor) -> torch.Tensor:
    pad_h = CFG.PAD_H - x.shape[-2]
    pad_w = CFG.PAD_W - x.shape[-1]
    return F.pad(x, (0, pad_w, 0, pad_h))


def crop_tensor_2d(x: torch.Tensor) -> torch.Tensor:
    return x[..., : CFG.H, : CFG.W]


def build_abs_prediction(delta_pred: torch.Tensor, last_cpm25: torch.Tensor) -> torch.Tensor:
    return delta_pred + last_cpm25.expand(-1, -1, -1, CFG.TIME_OUT)


def build_episode_mask(
    input_cpm25_seq_norm: torch.Tensor,
    y_true_norm: torch.Tensor,
    stats: Dict[str, Dict[str, float]],
) -> torch.Tensor:
    input_denorm = denorm_cpm25_torch(input_cpm25_seq_norm, stats)
    y_denorm = denorm_cpm25_torch(y_true_norm, stats)
    baseline = input_denorm.mean(dim=1)
    spread = input_denorm.std(dim=1, correction=0)
    threshold = baseline.unsqueeze(-1) + CFG.EPISODE_THRESHOLD_STD * spread.unsqueeze(-1)
    return y_denorm > threshold


def smape_tensor(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    denom = (0.5 * (pred.abs() + target.abs())).clamp_min(eps)
    return (pred - target).abs() / denom


def episode_correlation_loss(pred_denorm: torch.Tensor, true_denorm: torch.Tensor, episode_mask: torch.Tensor) -> torch.Tensor:
    corr_losses = []
    for t in range(CFG.TIME_OUT):
        mask_t = episode_mask[..., t]
        if mask_t.sum() < 4:
            continue
        p = pred_denorm[..., t][mask_t]
        y = true_denorm[..., t][mask_t]
        p = p - p.mean()
        y = y - y.mean()
        corr = (p * y).sum() / (p.norm() * y.norm()).clamp_min(1e-8)
        corr_losses.append(1.0 - corr)
    if corr_losses:
        return torch.stack(corr_losses).mean()
    return pred_denorm.new_tensor(0.0)


def compute_loss(
    delta_pred: torch.Tensor,
    y_true_norm: torch.Tensor,
    last_cpm25_norm: torch.Tensor,
    input_cpm25_seq_norm: torch.Tensor,
    x_static: torch.Tensor,
    stats: Dict[str, Dict[str, float]],
):
    abs_pred_norm = build_abs_prediction(delta_pred, last_cpm25_norm)
    delta_true_norm = y_true_norm - last_cpm25_norm.expand(-1, -1, -1, CFG.TIME_OUT)

    pred_denorm = denorm_cpm25_torch(abs_pred_norm, stats)
    true_denorm = denorm_cpm25_torch(y_true_norm, stats)
    episode_mask = build_episode_mask(input_cpm25_seq_norm, y_true_norm, stats)

    global_smape = smape_tensor(pred_denorm, true_denorm).mean()
    if bool(episode_mask.any()):
        episode_smape = smape_tensor(pred_denorm, true_denorm)[episode_mask].mean()
        episode_under = (true_denorm - pred_denorm).clamp_min(0.0)[episode_mask].mean()
    else:
        episode_smape = global_smape
        episode_under = pred_denorm.new_tensor(0.0)

    hotspot_prior = x_static[..., -1].unsqueeze(-1)
    hotspot_weight = 1.0 + CFG.HOTSPOT_WEIGHT_SCALE * hotspot_prior
    hotspot_l1 = (hotspot_weight * (pred_denorm - true_denorm).abs()).mean()
    delta_loss = F.huber_loss(delta_pred, delta_true_norm, delta=1.0)
    corr_loss = episode_correlation_loss(pred_denorm, true_denorm, episode_mask)

    total = (
        CFG.LOSS_WEIGHTS["delta"] * delta_loss
        + CFG.LOSS_WEIGHTS["global_smape"] * global_smape
        + CFG.LOSS_WEIGHTS["episode_smape"] * episode_smape
        + CFG.LOSS_WEIGHTS["episode_under"] * episode_under
        + CFG.LOSS_WEIGHTS["hotspot"] * hotspot_l1
        + CFG.LOSS_WEIGHTS["corr"] * corr_loss
    )

    parts = {
        "delta": float(delta_loss.detach().item()),
        "global_smape": float(global_smape.detach().item()),
        "episode_smape": float(episode_smape.detach().item()),
        "episode_under": float(episode_under.detach().item()),
        "hotspot": float(hotspot_l1.detach().item()),
        "corr": float(corr_loss.detach().item()),
    }
    return total, abs_pred_norm, episode_mask, parts


class MetricTracker:
    def __init__(self, time_out: int):
        self.time_out = time_out
        self.global_smape_sum = 0.0
        self.global_count = 0
        self.episode_smape_sum = 0.0
        self.episode_count = 0
        self.corr_n = np.zeros(time_out, dtype=np.float64)
        self.sum_pred = np.zeros(time_out, dtype=np.float64)
        self.sum_true = np.zeros(time_out, dtype=np.float64)
        self.sum_pred_sq = np.zeros(time_out, dtype=np.float64)
        self.sum_true_sq = np.zeros(time_out, dtype=np.float64)
        self.sum_pred_true = np.zeros(time_out, dtype=np.float64)

    def update(self, pred: np.ndarray, true: np.ndarray, episode_mask: np.ndarray) -> None:
        smape = np.abs(pred - true) / (0.5 * (np.abs(pred) + np.abs(true)) + 1e-3)
        self.global_smape_sum += float(smape.sum(dtype=np.float64))
        self.global_count += smape.size

        if episode_mask.any():
            episode_values = smape[episode_mask]
            self.episode_smape_sum += float(episode_values.sum(dtype=np.float64))
            self.episode_count += int(episode_values.size)

        for t in range(self.time_out):
            mask_t = episode_mask[..., t]
            if not np.any(mask_t):
                continue
            pred_t = pred[..., t][mask_t].astype(np.float64, copy=False)
            true_t = true[..., t][mask_t].astype(np.float64, copy=False)
            n = pred_t.size
            self.corr_n[t] += n
            self.sum_pred[t] += pred_t.sum()
            self.sum_true[t] += true_t.sum()
            self.sum_pred_sq[t] += np.square(pred_t).sum()
            self.sum_true_sq[t] += np.square(true_t).sum()
            self.sum_pred_true[t] += (pred_t * true_t).sum()

    def compute(self) -> Dict[str, float]:
        global_smape = self.global_smape_sum / max(self.global_count, 1)
        episode_smape = self.episode_smape_sum / self.episode_count if self.episode_count > 0 else global_smape

        corrs = []
        for t in range(self.time_out):
            n = self.corr_n[t]
            if n < 2:
                continue
            cov = self.sum_pred_true[t] - (self.sum_pred[t] * self.sum_true[t] / n)
            var_pred = self.sum_pred_sq[t] - (self.sum_pred[t] ** 2 / n)
            var_true = self.sum_true_sq[t] - (self.sum_true[t] ** 2 / n)
            denom = math.sqrt(max(var_pred, 0.0) * max(var_true, 0.0))
            corr = cov / (denom + 1e-12) if denom > 0 else 0.0
            corrs.append(float(np.clip(corr, -1.0, 1.0)))

        episode_corr = float(np.mean(corrs)) if corrs else 0.0
        norm_global = float(np.clip(1.0 - global_smape / 2.0, 0.0, 1.0))
        norm_episode = float(np.clip(1.0 - episode_smape / 2.0, 0.0, 1.0))
        norm_corr = float(np.clip((episode_corr + 1.0) / 2.0, 0.0, 1.0))
        score_proxy = float(np.mean([norm_global, norm_episode, norm_corr]))

        return {
            "global_smape": float(global_smape),
            "episode_smape": float(episode_smape),
            "episode_corr": float(episode_corr),
            "score_proxy": float(score_proxy),
        }

In [5]:
# =============================================================================
# CELL 4 - Model definition
# =============================================================================


class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        groups = min(8, out_ch)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, hidden, kernel_size=1)
        self.fc2 = nn.Conv2d(hidden, channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = self.pool(x)
        scale = F.gelu(self.fc1(scale))
        scale = torch.sigmoid(self.fc2(scale))
        return x * scale


class DecoderSEBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)
        self.se = SEBlock(out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.se(self.conv(x))


class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch: int, hidden_ch: int):
        super().__init__()
        self.hidden_ch = hidden_ch
        self.gates = nn.Conv2d(in_ch + hidden_ch, 4 * hidden_ch, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor, h: torch.Tensor, c: torch.Tensor):
        gates = self.gates(torch.cat([x, h], dim=1))
        i, f, o, g = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c

    def init_state(self, batch: int, height: int, width: int, device: torch.device):
        shape = (batch, self.hidden_ch, height, width)
        return torch.zeros(shape, device=device), torch.zeros(shape, device=device)


class StackedConvLSTM(nn.Module):
    def __init__(self, in_ch: int, hidden_ch: int, num_layers: int = 2):
        super().__init__()
        self.layers = nn.ModuleList(
            [ConvLSTMCell(in_ch if idx == 0 else hidden_ch, hidden_ch) for idx in range(num_layers)]
        )

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        batch, steps, _, height, width = seq.shape
        states = [layer.init_state(batch, height, width, seq.device) for layer in self.layers]
        for t in range(steps):
            x_t = seq[:, t]
            for layer_idx, layer in enumerate(self.layers):
                h, c = states[layer_idx]
                h, c = layer(x_t, h, c)
                states[layer_idx] = (h, c)
                x_t = h
        return states[-1][0]


class ResidualConvLSTMUNetV3(nn.Module):
    def __init__(self):
        super().__init__()
        in_ch = CFG.N_DYNAMIC + CFG.N_STATIC
        self.pool = nn.MaxPool2d(2)
        self.enc1 = DoubleConv(in_ch, CFG.MODEL_BASE)
        self.enc2 = DoubleConv(CFG.MODEL_BASE, CFG.MODEL_BASE * 2)
        self.enc3 = DoubleConv(CFG.MODEL_BASE * 2, CFG.MODEL_HIDDEN)
        self.bottleneck_in = DoubleConv(CFG.MODEL_HIDDEN, CFG.MODEL_HIDDEN)
        self.temporal = StackedConvLSTM(CFG.MODEL_HIDDEN, CFG.MODEL_HIDDEN, num_layers=2)
        self.spatial_attention = nn.Conv2d(CFG.MODEL_HIDDEN, 1, kernel_size=1)
        self.dropout = nn.Dropout2d(CFG.DROPOUT)
        self.up3 = nn.ConvTranspose2d(CFG.MODEL_HIDDEN, CFG.MODEL_HIDDEN, kernel_size=2, stride=2)
        self.dec3 = DecoderSEBlock(CFG.MODEL_HIDDEN * 2, CFG.MODEL_HIDDEN)
        self.up2 = nn.ConvTranspose2d(CFG.MODEL_HIDDEN, CFG.MODEL_BASE * 2, kernel_size=2, stride=2)
        self.dec2 = DecoderSEBlock(CFG.MODEL_BASE * 4, CFG.MODEL_BASE * 2)
        self.up1 = nn.ConvTranspose2d(CFG.MODEL_BASE * 2, CFG.MODEL_BASE, kernel_size=2, stride=2)
        self.dec1 = DecoderSEBlock(CFG.MODEL_BASE * 2, CFG.MODEL_BASE)
        self.head = nn.Conv2d(CFG.MODEL_BASE, CFG.TIME_OUT, kernel_size=1)

    def encode_step(self, x_step: torch.Tensor):
        e1 = self.enc1(x_step)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        bottleneck = self.bottleneck_in(self.pool(e3))
        return e1, e2, e3, bottleneck

    def forward(self, x_dyn: torch.Tensor, x_static: torch.Tensor) -> torch.Tensor:
        x_dyn = x_dyn.permute(0, 1, 4, 2, 3).contiguous()
        x_static = x_static.permute(0, 3, 1, 2).contiguous()

        sequence = []
        skip1 = skip2 = skip3 = None
        for t in range(x_dyn.shape[1]):
            x_step = torch.cat([x_dyn[:, t], x_static], dim=1)
            x_step = pad_tensor_2d(x_step)
            skip1, skip2, skip3, bottleneck = self.encode_step(x_step)
            sequence.append(bottleneck.unsqueeze(1))

        context = self.temporal(torch.cat(sequence, dim=1))
        attention = torch.sigmoid(self.spatial_attention(context))
        context = context * (1.0 + attention)
        context = self.dropout(context)

        d3 = self.up3(context)
        if d3.shape[-2:] != skip3.shape[-2:]:
            d3 = F.interpolate(d3, size=skip3.shape[-2:], mode="bilinear", align_corners=False)
        d3 = self.dec3(torch.cat([d3, skip3], dim=1))

        d2 = self.up2(d3)
        if d2.shape[-2:] != skip2.shape[-2:]:
            d2 = F.interpolate(d2, size=skip2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([d2, skip2], dim=1))

        d1 = self.up1(d2)
        if d1.shape[-2:] != skip1.shape[-2:]:
            d1 = F.interpolate(d1, size=skip1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([d1, skip1], dim=1))

        out = crop_tensor_2d(self.head(d1))
        return out.permute(0, 2, 3, 1).contiguous()


def build_model() -> nn.Module:
    return ResidualConvLSTMUNetV3().to(DEVICE)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def autocast_context():
    if AMP_ENABLED:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def build_grad_scaler():
    if AMP_ENABLED and hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda", enabled=True)
        except TypeError:
            try:
                return torch.amp.GradScaler(enabled=True)
            except TypeError:
                pass
    return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = CFG.EMA_DECAY):
        self.decay = decay
        self.ema = deepcopy(model).to(DEVICE)
        self.ema.eval()
        for param in self.ema.parameters():
            param.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        src = model.state_dict()
        tgt = self.ema.state_dict()
        for key, value in tgt.items():
            src_value = src[key].detach()
            if torch.is_floating_point(value):
                value.mul_(self.decay).add_(src_value, alpha=1.0 - self.decay)
            else:
                value.copy_(src_value)

In [6]:
# =============================================================================
# CELL 5 - Training, validation, tune, final-refit, inference
# =============================================================================


def make_scheduler(optimizer: torch.optim.Optimizer, total_epochs: int, warmup_epochs: int):
    def lr_lambda(epoch: int) -> float:
        if total_epochs <= 1:
            return 1.0
        if epoch < warmup_epochs:
            return float(epoch + 1) / max(warmup_epochs, 1)
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs - 1, 1)
        progress = min(max(progress, 0.0), 1.0)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def run_shape_sanity(model: nn.Module, loader: DataLoader) -> None:
    x_dyn, x_static, y, last_cpm25, _ = next(iter(loader))
    x_dyn = x_dyn.to(DEVICE, non_blocking=True)
    x_static = x_static.to(DEVICE, non_blocking=True)
    y = y.to(DEVICE, non_blocking=True)
    last_cpm25 = last_cpm25.to(DEVICE, non_blocking=True)
    with torch.no_grad():
        with autocast_context():
            delta_pred = model(x_dyn, x_static)
            abs_pred = build_abs_prediction(delta_pred, last_cpm25)
    assert delta_pred.shape == (x_dyn.shape[0], CFG.H, CFG.W, CFG.TIME_OUT), delta_pred.shape
    assert abs_pred.shape == y.shape, (abs_pred.shape, y.shape)
    print("Shape sanity check passed")


def validate_epoch(model: nn.Module, loader: DataLoader, stats: Dict[str, Dict[str, float]], limit_batches: int | None = None):
    model.eval()
    tracker = MetricTracker(CFG.TIME_OUT)
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for batch_idx, (x_dyn, x_static, y, last_cpm25, input_cpm25_seq) in enumerate(loader):
            if limit_batches is not None and batch_idx >= limit_batches:
                break

            x_dyn = x_dyn.to(DEVICE, non_blocking=True)
            x_static = x_static.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            last_cpm25 = last_cpm25.to(DEVICE, non_blocking=True)
            input_cpm25_seq = input_cpm25_seq.to(DEVICE, non_blocking=True)

            with autocast_context():
                delta_pred = model(x_dyn, x_static)
                loss, abs_pred, episode_mask, _ = compute_loss(
                    delta_pred,
                    y,
                    last_cpm25,
                    input_cpm25_seq,
                    x_static,
                    stats,
                )

            total_loss += float(loss.detach().item())
            total_batches += 1

            pred_denorm = denorm_cpm25_torch(abs_pred.float(), stats).cpu().numpy()
            true_denorm = denorm_cpm25_torch(y.float(), stats).cpu().numpy()
            tracker.update(pred_denorm, true_denorm, episode_mask.cpu().numpy().astype(bool))

    metrics = tracker.compute()
    metrics["val_loss"] = total_loss / max(total_batches, 1)
    return metrics


def cleanup_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)
    for name in os.listdir(path):
        full = os.path.join(path, name)
        if os.path.isfile(full):
            try:
                os.remove(full)
            except OSError:
                pass


def save_topk_aliases(topk_entries: List[dict], target_dir: str) -> List[str]:
    final_paths = []
    for rank, entry in enumerate(topk_entries, start=1):
        target = os.path.join(target_dir, f"top{rank}.pt")
        shutil.copyfile(entry["path"], target)
        final_paths.append(target)
    return final_paths


def train_tune_model(
    train_dataset: Dataset,
    val_dataset: Dataset,
    stats: Dict[str, Dict[str, float]],
) -> dict:
    cleanup_dir(CFG.TUNE_CKPT_DIR)

    train_loader = make_loader(train_dataset, CFG.TRAIN_BATCH, shuffle=True)
    val_loader = make_loader(val_dataset, CFG.TRAIN_BATCH, shuffle=False)

    model = build_model()
    ema = ModelEMA(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = make_scheduler(optimizer, CFG.TUNE_EPOCHS if not DEBUG else 1, CFG.WARMUP_EPOCHS)
    scaler = build_grad_scaler()
    run_shape_sanity(model, train_loader)

    best_score = -float("inf")
    best_epoch = 0
    stale_epochs = 0
    history = []
    topk_entries = []
    debug_train_limit = 2 if DEBUG else None
    debug_val_limit = 2 if DEBUG else None

    print(f"Trainable parameters: {count_parameters(model) / 1e6:.2f}M")

    total_epochs = 1 if DEBUG else CFG.TUNE_EPOCHS
    for epoch in range(total_epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_start = time.time()
        total_loss = 0.0
        total_batches = 0
        oom_skips = 0
        limit_steps = min(len(train_loader), debug_train_limit) if debug_train_limit is not None else len(train_loader)

        for batch_idx, (x_dyn, x_static, y, last_cpm25, input_cpm25_seq) in enumerate(train_loader):
            if debug_train_limit is not None and batch_idx >= debug_train_limit:
                break

            x_dyn = x_dyn.to(DEVICE, non_blocking=True)
            x_static = x_static.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            last_cpm25 = last_cpm25.to(DEVICE, non_blocking=True)
            input_cpm25_seq = input_cpm25_seq.to(DEVICE, non_blocking=True)

            try:
                with autocast_context():
                    delta_pred = model(x_dyn, x_static)
                    loss, _, _, _ = compute_loss(
                        delta_pred,
                        y,
                        last_cpm25,
                        input_cpm25_seq,
                        x_static,
                        stats,
                    )
                    scaled_loss = loss / CFG.GRAD_ACCUM

                if AMP_ENABLED:
                    scaler.scale(scaled_loss).backward()
                else:
                    scaled_loss.backward()

                should_step = ((batch_idx + 1) % CFG.GRAD_ACCUM == 0) or ((batch_idx + 1) == limit_steps)
                if should_step:
                    if AMP_ENABLED:
                        scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    if AMP_ENABLED:
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    ema.update(model)

                total_loss += float(loss.detach().item())
                total_batches += 1
            except RuntimeError as exc:
                if "out of memory" not in str(exc).lower():
                    raise
                oom_skips += 1
                optimizer.zero_grad(set_to_none=True)
                gc.collect()
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
                print(f"[tune] skipped OOM batch {batch_idx}")

        scheduler.step()
        val_metrics = validate_epoch(ema.ema, val_loader, stats, limit_batches=debug_val_limit)

        epoch_log = {
            "phase": "tune",
            "epoch": epoch + 1,
            "train_loss": round(total_loss / max(total_batches, 1), 6),
            "lr": round(float(optimizer.param_groups[0]["lr"]), 8),
            "minutes": round((time.time() - epoch_start) / 60.0, 2),
            "oom_skips": oom_skips,
            "val_loss": round(val_metrics["val_loss"], 6),
            "global_smape": round(val_metrics["global_smape"], 6),
            "episode_smape": round(val_metrics["episode_smape"], 6),
            "episode_corr": round(val_metrics["episode_corr"], 6),
            "score_proxy": round(val_metrics["score_proxy"], 6),
        }
        history.append(epoch_log)
        print(f"[tune] epoch {epoch + 1:02d} | {epoch_log}")

        candidate_path = os.path.join(CFG.TUNE_CKPT_DIR, f"epoch_{epoch + 1:02d}.pt")
        torch.save(
            {
                "epoch": epoch + 1,
                "score_proxy": val_metrics["score_proxy"],
                "model_state": ema.ema.state_dict(),
                "history": history,
            },
            candidate_path,
        )
        topk_entries.append({"path": candidate_path, "score": float(val_metrics["score_proxy"]), "epoch": epoch + 1})
        topk_entries.sort(key=lambda item: item["score"], reverse=True)
        while len(topk_entries) > TOPK_CKPTS:
            removed = topk_entries.pop(-1)
            if os.path.exists(removed["path"]):
                os.remove(removed["path"])

        if val_metrics["score_proxy"] > best_score:
            best_score = float(val_metrics["score_proxy"])
            best_epoch = epoch + 1
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= CFG.PATIENCE:
                print(f"[tune] early stopping at epoch {epoch + 1}")
                break

    save_json(CFG.LOG_PATH, history)
    final_top_paths = save_topk_aliases(topk_entries, CFG.TUNE_CKPT_DIR)

    del model, ema, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    weighted_epoch = sum(entry["epoch"] * entry["score"] for entry in topk_entries) / max(sum(entry["score"] for entry in topk_entries), 1e-8)
    suggested_final_epoch = int(round(weighted_epoch)) + CFG.FINAL_EXTRA_EPOCHS
    suggested_final_epoch = max(4, min(suggested_final_epoch, 12))

    return {
        "best_score": best_score,
        "best_epoch": best_epoch,
        "suggested_final_epoch": suggested_final_epoch,
        "top_paths": final_top_paths,
        "history": history,
    }


def train_final_full_model(
    train_dataset: Dataset,
    stats: Dict[str, Dict[str, float]],
    epochs: int,
    seed: int,
    save_epochs: List[int],
) -> List[str]:
    train_loader = make_loader(train_dataset, CFG.TRAIN_BATCH, shuffle=True)
    set_seed(seed)
    model = build_model()
    ema = ModelEMA(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.FINAL_LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = make_scheduler(optimizer, epochs if not DEBUG else 1, CFG.WARMUP_EPOCHS)
    scaler = build_grad_scaler()

    saved_paths = []
    total_epochs = 1 if DEBUG else epochs
    print(f"[final seed={seed}] epochs={total_epochs} save_epochs={save_epochs}")

    for epoch in range(total_epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_start = time.time()
        total_loss = 0.0
        total_batches = 0

        for batch_idx, (x_dyn, x_static, y, last_cpm25, input_cpm25_seq) in enumerate(train_loader):
            x_dyn = x_dyn.to(DEVICE, non_blocking=True)
            x_static = x_static.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            last_cpm25 = last_cpm25.to(DEVICE, non_blocking=True)
            input_cpm25_seq = input_cpm25_seq.to(DEVICE, non_blocking=True)

            with autocast_context():
                delta_pred = model(x_dyn, x_static)
                loss, _, _, _ = compute_loss(
                    delta_pred,
                    y,
                    last_cpm25,
                    input_cpm25_seq,
                    x_static,
                    stats,
                )
                scaled_loss = loss / CFG.GRAD_ACCUM

            if AMP_ENABLED:
                scaler.scale(scaled_loss).backward()
            else:
                scaled_loss.backward()

            should_step = ((batch_idx + 1) % CFG.GRAD_ACCUM == 0) or ((batch_idx + 1) == len(train_loader))
            if should_step:
                if AMP_ENABLED:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if AMP_ENABLED:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)

            total_loss += float(loss.detach().item())
            total_batches += 1

        scheduler.step()
        epoch_num = epoch + 1
        print(
            f"[final seed={seed}] epoch {epoch_num:02d} | "
            f"train_loss={total_loss / max(total_batches, 1):.6f} | "
            f"lr={optimizer.param_groups[0]['lr']:.7f} | "
            f"minutes={(time.time() - epoch_start) / 60.0:.2f}"
        )

        if epoch_num in save_epochs:
            ckpt_path = os.path.join(CFG.FINAL_CKPT_DIR, f"seed{seed}_epoch{epoch_num:02d}.pt")
            torch.save(
                {
                    "epoch": epoch_num,
                    "seed": seed,
                    "model_state": ema.ema.state_dict(),
                },
                ckpt_path,
            )
            saved_paths.append(ckpt_path)
            print(f"[final seed={seed}] saved snapshot -> {ckpt_path}")

    del model, ema, optimizer, scheduler, scaler, train_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return saved_paths


def load_eval_model(checkpoint_path: str) -> nn.Module:
    model = build_model()
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model


@torch.no_grad()
def infer_checkpoint(checkpoint_path: str, dataset: Dataset, stats: Dict[str, Dict[str, float]]) -> np.ndarray:
    loader = make_loader(dataset, CFG.INFER_BATCH, shuffle=False)
    model = load_eval_model(checkpoint_path)
    preds = np.zeros((len(dataset), CFG.H, CFG.W, CFG.TIME_OUT), dtype=np.float32)
    debug_limit = 1 if DEBUG else None

    for batch_idx, (x_dyn, x_static, last_cpm25, idx) in enumerate(
        tqdm(loader, desc=os.path.basename(checkpoint_path))
    ):
        if debug_limit is not None and batch_idx >= debug_limit:
            break

        x_dyn = x_dyn.to(DEVICE, non_blocking=True)
        x_static = x_static.to(DEVICE, non_blocking=True)
        last_cpm25 = last_cpm25.to(DEVICE, non_blocking=True)

        with autocast_context():
            delta_pred = model(x_dyn, x_static)
            abs_pred_norm = build_abs_prediction(delta_pred, last_cpm25)

        pred = denorm_cpm25_torch(abs_pred_norm.float(), stats).cpu().numpy()
        preds[idx.numpy()] = pred

    del model, loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return preds


def get_default_inference_paths() -> List[str]:
    paths = []
    for name in sorted(os.listdir(CFG.FINAL_CKPT_DIR)):
        if name.endswith(".pt"):
            paths.append(os.path.join(CFG.FINAL_CKPT_DIR, name))
    if paths:
        return paths

    for rank in range(1, TOPK_CKPTS + 1):
        path = os.path.join(CFG.TUNE_CKPT_DIR, f"top{rank}.pt")
        if os.path.exists(path):
            paths.append(path)
    if not paths:
        raise FileNotFoundError("No checkpoints available for inference.")
    return paths


def run_inference(stats: Dict[str, Dict[str, float]], hotspot_prior: np.ndarray, checkpoint_paths: List[str] | None = None) -> None:
    dataset = TestStreamingDataset(stats, hotspot_prior)
    checkpoint_paths = checkpoint_paths or get_default_inference_paths()
    print("Inference checkpoints:")
    for path in checkpoint_paths:
        print(f"  {path}")

    ensemble = np.zeros((len(dataset), CFG.H, CFG.W, CFG.TIME_OUT), dtype=np.float32)
    for checkpoint_path in checkpoint_paths:
        ensemble += infer_checkpoint(checkpoint_path, dataset, stats)

    ensemble /= max(len(checkpoint_paths), 1)
    ensemble = np.clip(ensemble, 0.0, None).astype(np.float32)

    if DEBUG:
        print(f"DEBUG inference complete: shape={ensemble.shape}")
    else:
        assert ensemble.shape == (218, CFG.H, CFG.W, CFG.TIME_OUT), ensemble.shape
        np.save(CFG.PREDS_PATH, ensemble)
        print(f"Saved predictions to {CFG.PREDS_PATH}")
        print(
            f"preds.npy -> shape={ensemble.shape}, dtype={ensemble.dtype}, "
            f"min={ensemble.min():.3f}, max={ensemble.max():.3f}, mean={ensemble.mean():.3f}"
        )

In [7]:
# =============================================================================
# CELL 6 - Main pipeline
# =============================================================================


def prepare_tune_artifacts():
    stats_tune = compute_stats(TRAIN_CUTOFFS)
    save_json(CFG.STATS_TUNE_PATH, stats_tune)
    month_cache_tune = build_month_cache(stats_tune)
    hotspot_prior_tune = build_hotspot_prior(month_cache_tune, TRAIN_ITEMS, stats_tune)
    np.save(CFG.HOTSPOT_TUNE_PATH, hotspot_prior_tune.astype(np.float32))
    return stats_tune, month_cache_tune, hotspot_prior_tune


def prepare_full_artifacts():
    stats_full = compute_stats(None)
    save_json(CFG.STATS_FULL_PATH, stats_full)
    month_cache_full = build_month_cache(stats_full)
    hotspot_prior_full = build_hotspot_prior(month_cache_full, ALL_ITEMS, stats_full)
    np.save(CFG.HOTSPOT_FULL_PATH, hotspot_prior_full.astype(np.float32))
    return stats_full, month_cache_full, hotspot_prior_full


def load_saved_artifacts(stats_path: str, hotspot_path: str):
    with open(stats_path, "r") as handle:
        stats = json.load(handle)
    hotspot_prior = np.load(hotspot_path).astype(np.float32)
    return stats, hotspot_prior


def main() -> None:
    tune_result = None

    if RUN_TUNE:
        stats_tune, month_cache_tune, hotspot_prior_tune = prepare_tune_artifacts()
        train_dataset = WindowDataset(month_cache_tune, TRAIN_ITEMS, hotspot_prior_tune)
        val_dataset = WindowDataset(month_cache_tune, VAL_ITEMS, hotspot_prior_tune)

        tune_result = train_tune_model(train_dataset, val_dataset, stats_tune)
        print(
            f"Tuning complete | best_epoch={tune_result['best_epoch']} | "
            f"best_score={tune_result['best_score']:.6f} | "
            f"suggested_final_epoch={tune_result['suggested_final_epoch']}"
        )

        del train_dataset, val_dataset, month_cache_tune
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    else:
        stats_tune, hotspot_prior_tune = load_saved_artifacts(CFG.STATS_TUNE_PATH, CFG.HOTSPOT_TUNE_PATH)

    final_checkpoint_paths = []

    if RUN_FINAL_TRAIN:
        stats_full, month_cache_full, hotspot_prior_full = prepare_full_artifacts()
        full_dataset = WindowDataset(month_cache_full, ALL_ITEMS, hotspot_prior_full)

        if tune_result is None:
            tuned_epoch = CFG.FINAL_EPOCHS if CFG.FINAL_EPOCHS is not None else 8
        else:
            tuned_epoch = tune_result["suggested_final_epoch"]
        final_epochs = CFG.FINAL_EPOCHS if CFG.FINAL_EPOCHS is not None else tuned_epoch
        snapshot_epochs = sorted(
            {
                min(max(final_epochs + offset, 1), final_epochs)
                for offset in CFG.FINAL_SNAPSHOT_OFFSETS
            }
        )
        snapshot_epochs = [ep for ep in snapshot_epochs if ep <= final_epochs]
        print(f"Final full-data train epochs={final_epochs} | snapshot_epochs={snapshot_epochs}")

        cleanup_dir(CFG.FINAL_CKPT_DIR)
        for seed in CFG.FINAL_SEEDS:
            final_checkpoint_paths.extend(
                train_final_full_model(
                    train_dataset=full_dataset,
                    stats=stats_full,
                    epochs=final_epochs,
                    seed=seed,
                    save_epochs=snapshot_epochs,
                )
            )

        del full_dataset, month_cache_full
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    else:
        stats_full, hotspot_prior_full = load_saved_artifacts(CFG.STATS_FULL_PATH, CFG.HOTSPOT_FULL_PATH)

    if RUN_INFER:
        if RUN_FINAL_TRAIN:
            run_inference(stats_full, hotspot_prior_full, checkpoint_paths=final_checkpoint_paths)
        elif RUN_TUNE:
            run_inference(stats_tune, hotspot_prior_tune)
        else:
            stats_full, hotspot_prior_full = load_saved_artifacts(CFG.STATS_FULL_PATH, CFG.HOTSPOT_FULL_PATH)
            run_inference(stats_full, hotspot_prior_full)

    print("Fine-tuned latest notebook complete.")


if __name__ == "__main__":
    main()

Computing tune stats:   0%|          | 0/4 [00:00<?, ?it/s]

Cached APRIL_16: (715, 140, 124, 20) | 0.99 GB
Cached JULY_16: (739, 140, 124, 20) | 1.03 GB
Cached OCT_16: (739, 140, 124, 20) | 1.03 GB
Cached DEC_16: (739, 140, 124, 20) | 1.03 GB
Cache RAM footprint: 4.07 GB


Building hotspot prior:   0%|          | 0/2576 [00:00<?, ?it/s]

Shape sanity check passed
Trainable parameters: 5.88M
[tune] epoch 01 | {'phase': 'tune', 'epoch': 1, 'train_loss': 2.562186, 'lr': 0.0008, 'minutes': 3.06, 'oom_skips': 0, 'val_loss': 2.433693, 'global_smape': 0.288004, 'episode_smape': 0.203158, 'episode_corr': 0.951643, 'score_proxy': 0.91008}
[tune] epoch 02 | {'phase': 'tune', 'epoch': 2, 'train_loss': 2.15555, 'lr': 0.00076955, 'minutes': 2.72, 'oom_skips': 0, 'val_loss': 2.13855, 'global_smape': 0.257832, 'episode_smape': 0.17575, 'episode_corr': 0.952576, 'score_proxy': 0.919832}
[tune] epoch 03 | {'phase': 'tune', 'epoch': 3, 'train_loss': 1.9963, 'lr': 0.00068284, 'minutes': 2.72, 'oom_skips': 0, 'val_loss': 1.999908, 'global_smape': 0.243128, 'episode_smape': 0.167183, 'episode_corr': 0.955483, 'score_proxy': 0.924195}
[tune] epoch 04 | {'phase': 'tune', 'epoch': 4, 'train_loss': 1.864192, 'lr': 0.00055307, 'minutes': 2.73, 'oom_skips': 0, 'val_loss': 1.902447, 'global_smape': 0.235225, 'episode_smape': 0.164124, 'episode_co

Computing full-data stats:   0%|          | 0/4 [00:00<?, ?it/s]

...

seed42_epoch08.pt:   0%|          | 0/55 [00:00<?, ?it/s]

seed42_epoch09.pt:   0%|          | 0/55 [00:00<?, ?it/s]

seed3407_epoch08.pt:   0%|          | 0/55 [00:00<?, ?it/s]

seed3407_epoch09.pt:   0%|          | 0/55 [00:00<?, ?it/s]

Saved predictions to /kaggle/working/preds.npy
preds.npy -> shape=(218, 140, 124, 16), dtype=float32, min=0.000, max=1523.832, mean=36.559
Fine-tuned latest notebook complete.
